# Ablation Study S1–S8

## How to run this notebook (2-pass workflow)

**This notebook is designed to be run in two passes.**

#### Pass 1 — Build the Error Bank (S1 only)

1. Set: GENERATE_ERROR_BANK = True


2. Run Section 1 only (up to and including the “Generate Error Bank” cell).

      This creates/updates:

      error_bank_v1.csv (raw error bank, used later by S2–S8)

      S1 retrieval result CSVs used to define failures

3. After Pass 1, stop and run the **QUR** rewrite notebook.

#### Pass 2 — Run the full ablation (S2–S8)

1. Run Section 2 (S2–S8)

# SECTION 1

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ==========================================================
# 0) CONFIG
# ==========================================================
import os, re, math, json, time
from typing import Any, Dict, List, Tuple, Optional, Iterable
from collections import defaultdict
import numpy as np
import pandas as pd

pd.options.display.float_format = '{:.4f}'.format

# ---- Control Flags ----
GENERATE_ERROR_BANK = True  # Set to True for first run, then False

# ---- Paths (Updated for ComplianceGPT_v2 Drive Structure) ----
DRIVE_ROOT = "/content/drive/MyDrive/ComplianceGPT_v2"

# 1. Catalog Paths (CCS)
REV5_CATALOG_PATH = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = f"{DRIVE_ROOT}/data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# 2. Gold Standard Datasets
REV5_GOLD_CSV = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev5_gold-set_100q.csv"
REV4_GOLD_CSV = f"{DRIVE_ROOT}/data/gold_standard_datasets/nist800-53/nist_sp800-53_rev4_gold-set_36q.csv"

# 3. Error Bank
ERROR_BANK_CSV = f"{DRIVE_ROOT}/data/error_bank/error_bank_v1.csv"

# 4. QUR Rewrites
QUR_CSV_REV5 = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_rev5.csv"
QUR_CSV_REV4 = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_rev4.csv"
QUR_CSV_ERRB = f"{DRIVE_ROOT}/data/qur_outputs/qur_rewrites_error_bank.csv"

# 5. Output Directory
BASE_OUT_DIR = f"{DRIVE_ROOT}/experiments/retriever_ablation/ablation_outputs"
S1_DIR = os.path.join(BASE_OUT_DIR, "system_1_bm25")
S2_DIR = os.path.join(BASE_OUT_DIR, "system_2_dense")
S3_DIR = os.path.join(BASE_OUT_DIR, "system_3_rewrite_only")
S4_DIR = os.path.join(BASE_OUT_DIR, "system_4_qur_rrf")
S5_DIR = os.path.join(BASE_OUT_DIR, "system_5_hybrid_rrf")
S6_DIR = os.path.join(BASE_OUT_DIR, "system_6_hybrid_rerank")
S7_DIR = os.path.join(BASE_OUT_DIR, "system_7_compliance_gpt")
S8_DIR = os.path.join(BASE_OUT_DIR, "system_8_adaptive")

for d in [S1_DIR, S2_DIR, S3_DIR, S4_DIR, S5_DIR, S6_DIR, S7_DIR, S8_DIR]:
    os.makedirs(d, exist_ok=True)

# Hyperparams
TOP_N = 10
BM25_K1 = 1.5
BM25_B = 0.75
DENSE_MODEL_ID = "intfloat/e5-small-v2"
RERANKER_MODEL_ID = "BAAI/bge-reranker-base"
CANDIDATE_SET_SIZE = 50
RRF_K_DEFAULT = 60
S2_CLAUSE_TOP_K = 300

# S8 Knobs
S8_CONF_THRESHOLD = 0.70
S8_MARGIN_RATIO_STRONG = 0.12
S8_MAX_REWRITES = 3
S8_REWRITE_WEIGHT = 0.25
S8_REWRITE_JACCARD_MIN = 0.15
S8_RERANK_ALPHA = 0.65
S8_HEAVY_MIN_MARGIN_RATIO = 0.06

print(" Config loaded.")

 Config loaded.


In [3]:
# ==========================================================
# 1) TEXT + CONTROL-ID NORMALIZATION & JSONL LOADERS
# ==========================================================
PUNCT_RE = re.compile(r"[^0-9A-Za-z_\s]+")
WS_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def normalize_text(text: str) -> str:
    if not isinstance(text, str): text = "" if pd.isna(text) else str(text)
    s = text.lower().replace('-', ' ')
    s = PUNCT_RE.sub(' ', s)
    return WS_RE.sub(' ', s).strip()

def tokenize(text: str) -> List[str]:
    return [t.lower() for t in TOKEN_RE.findall(str(text))]

def normalize_control_id(control_id: str) -> str:
    if not isinstance(control_id, str): return ""
    return control_id.strip().upper().replace("_", "-")

def control_id_aliases(control_id: str) -> str:
    cid = normalize_control_id(control_id)
    return f"{cid} {cid.replace('-', '')} {cid.replace('-', ' ')}"

def load_clause_records_jsonl(jsonl_path: str, keep_kinds=("smt", "gdn")) -> List[Dict]:
    out = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            rec = json.loads(line)
            if keep_kinds and str(rec.get("kind","")).lower() not in keep_kinds: continue

            txt = str(rec.get("text","")).strip()
            if not txt: continue

            out.append({
                "clause_id": str(rec.get("id","")).strip(),
                "control_id": normalize_control_id(rec.get("control_id") or rec.get("control") or ""),
                "title": str(rec.get("title","")).strip(),
                "text": txt,
                "kind": str(rec.get("kind","")).lower()
            })
    return out

def build_control_docs_from_clauses(records):
    by_ctl = defaultdict(list)
    titles = {}
    for r in records:
        cid = r["control_id"]
        if not cid: continue
        by_ctl[cid].append(r["text"])
        if r["title"]: titles[cid] = r["title"]

    docs = []
    for cid, parts in by_ctl.items():
        header = [control_id_aliases(cid)]
        if titles.get(cid): header.append(titles[cid])
        full_text = "\n".join(header + parts)
        docs.append({"control_id": cid, "text": full_text})
    return docs

def build_control_fallback_text_map(records):
    # For reranker fallback
    by_ctl = defaultdict(list)
    for r in records:
        cid = r["control_id"]
        if cid: by_ctl[cid].append(r["text"])
    return {k: "\n".join(v[:8]) for k,v in by_ctl.items()}

In [4]:
# ==========================================================
# 2) BM25 + METRICS
# ==========================================================
class BM25Okapi:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N else 0.0

        self.tf = []
        df = {}
        for doc in corpus_tokens:
            tf_doc = {}
            for t in doc:
                tf_doc[t] = tf_doc.get(t, 0) + 1
            self.tf.append(tf_doc)
            for t in tf_doc:
                df[t] = df.get(t, 0) + 1

        self.idf = {t: math.log((self.N - freq + 0.5)/(freq + 0.5) + 1) for t, freq in df.items()}

    def get_scores(self, query_tokens):
        scores = np.zeros(self.N, dtype=np.float32)
        for t in query_tokens:
            if t not in self.idf: continue
            idf = self.idf[t]
            for i, tf_doc in enumerate(self.tf):
                f = tf_doc.get(t, 0)
                if f:
                    num = f * (self.k1 + 1)
                    den = f + self.k1 * (1 - self.b + self.b * (self.doc_len[i]/self.avgdl))
                    scores[i] += idf * (num/den)
        return scores

    def get_top_n(self, query_tokens, n=10):
        scores = self.get_scores(query_tokens)
        if n >= len(scores): return np.argsort(-scores).tolist()
        idx = np.argpartition(-scores, n-1)[:n]
        return idx[np.argsort(-scores[idx])].tolist()

def build_bm25(docs):
    ids = [d["control_id"] for d in docs]
    toks = [tokenize(normalize_text(d["text"])) for d in docs]
    return BM25Okapi(toks, k1=BM25_K1, b=BM25_B), ids

In [5]:
# ==========================================================
# 3) DENSE RETRIEVER SETUP
# ==========================================================
try:
    import faiss
    import sentence_transformers
except:
    !pip install -q sentence-transformers faiss-cpu

import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

class DenseIndex:
    def __init__(self, model_id):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Loading Dense Model {model_id} on {self.device}...")
        self.model = SentenceTransformer(model_id, device=self.device)
        self.index = None
        self.ids = []

    def build(self, texts, ids):
        self.ids = ids
        emb = self.model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(emb)

    def search(self, query, top_k):
        q_emb = self.model.encode([query], normalize_embeddings=True)
        scores, idxs = self.index.search(q_emb, top_k)
        return [(self.ids[i], float(s)) for i, s in zip(idxs[0], scores[0]) if i != -1]

# Clause-to-Control Adapter
class ClauseDenseControlAdapter:
    def __init__(self, dense_idx, clause_map, text_map):
        self.idx = dense_idx
        self.cmap = clause_map
        self.tmap = text_map
        self.last_best_clause = {} # query-specific cache

    def search(self, query, top_k=50):
        # Search more clauses to ensure we get enough unique controls
        hits = self.idx.search(query, top_k=top_k*5)
        best_score = {}
        best_clause = {}

        for cid, score in hits:
            ctl = self.cmap.get(cid)
            if not ctl: continue
            ctl = normalize_control_id(ctl)
            if score > best_score.get(ctl, -1.0):
                best_score[ctl] = score
                best_clause[ctl] = cid

        self.last_best_clause = best_clause
        ranked = sorted(best_score.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

    def best_clause_text_for_control(self, ctl_id):
        cid = normalize_control_id(ctl_id)
        clid = self.last_best_clause.get(cid)
        return self.tmap.get(clid) if clid else None

def build_dense_retriever(jsonl_path):
    recs = load_clause_records_jsonl(jsonl_path, keep_kinds={"smt", "gdn"})
    ids, texts = [], []
    cmap, tmap = {}, {}
    for r in recs:
        header = f"{r['control_id']} {r['title']} ({r['kind']})"
        full = f"{header}\n{r['text']}"
        ids.append(r["clause_id"])
        texts.append(f"passage: {full}") # e5 prefix
        cmap[r["clause_id"]] = r["control_id"]
        tmap[r["clause_id"]] = full

    idx = DenseIndex(DENSE_MODEL_ID)
    idx.build(texts, ids)
    return ClauseDenseControlAdapter(idx, cmap, tmap)

In [6]:
# ==========================================================
# 4) LOAD DATA & BUILD INDICES
# ==========================================================
print("--- Loading Gold Sets ---")
rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding='ISO-8859-1')
rev4_gold = pd.read_csv(REV4_GOLD_CSV, encoding='ISO-8859-1')
# Note: Error Bank is NOT loaded here yet. We will regenerate it.

qur_rev5_df = pd.read_csv(QUR_CSV_REV5, encoding='ISO-8859-1')
qur_rev4_df = pd.read_csv(QUR_CSV_REV4, encoding='ISO-8859-1')
# Use combined QUR for error bank fallback
qur_errb_df = pd.concat([qur_rev5_df, qur_rev4_df])

print("--- Building BM25 Indices ---")
rev5_c_recs = load_clause_records_jsonl(REV5_CATALOG_PATH)
rev4_c_recs = load_clause_records_jsonl(REV4_CATALOG_PATH)
bm25_rev5, rev5_ids = build_bm25(build_control_docs_from_clauses(rev5_c_recs))
bm25_rev4, rev4_ids = build_bm25(build_control_docs_from_clauses(rev4_c_recs))

print("--- Building Dense Indices (Clause-Level) ---")
dense_retriever_rev5 = build_dense_retriever(REV5_CATALOG_PATH)
dense_retriever_rev4 = build_dense_retriever(REV4_CATALOG_PATH)

print("--- Loading Reranker ---")
cross_encoder = CrossEncoder(RERANKER_MODEL_ID, device='cuda' if torch.cuda.is_available() else 'cpu')

# Fallback text maps
rev5_id_to_text = build_control_fallback_text_map(rev5_c_recs)
rev4_id_to_text = build_control_fallback_text_map(rev4_c_recs)

print(" System Ready.")

--- Loading Gold Sets ---
--- Building BM25 Indices ---
--- Building Dense Indices (Clause-Level) ---
Loading Dense Model intfloat/e5-small-v2 on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

--- Loading Reranker ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 System Ready.


In [7]:
# ==========================================================
# 5) SYSTEM LOGIC DEFINITIONS (S1-S8) — UNIFIED & PERFECTED
# ==========================================================
import numpy as np
import pandas as pd
import json

# --- Helpers ---
def find_rank(gold, ranked_list, top_n):
    g = normalize_control_id(gold)
    for i, c in enumerate(ranked_list[:top_n], 1):
        if normalize_control_id(c) == g:
            return i
    return 0

def strict_clean(s: str) -> str:
    """Strict alnum-only key for stable exact matching."""
    return "".join(c.lower() for c in str(s) if c.isalnum())

def jaccard_overlap(a: str, b: str) -> float:
    """Token Jaccard overlap between two queries (0..1)."""
    A = set(tokenize(normalize_text(a)))
    B = set(tokenize(normalize_text(b)))
    if not A or not B: return 0.0
    return len(A & B) / float(len(A | B))

def normalize_scores_minmax(x: np.ndarray) -> np.ndarray:
    """Min-max normalize to [0,1]. Returns zeros if constant."""
    if x.size == 0: return x
    mn, mx = float(np.min(x)), float(np.max(x))
    if mx - mn < 1e-12: return np.zeros_like(x, dtype=np.float32)
    return ((x - mn) / (mx - mn)).astype(np.float32)

def get_rewrites(query, qur_df, max_rewrites=5):
    """Legacy helper for S3: Return rewrites using strict matching."""
    if qur_df is None or qur_df.empty: return []
    if 'strict_key' not in qur_df.columns and 'original_query' in qur_df.columns:
        qur_df['strict_key'] = qur_df['original_query'].apply(strict_clean)

    q_key = strict_clean(query)
    matches = qur_df[qur_df['strict_key'] == q_key] if 'strict_key' in qur_df.columns else pd.DataFrame()

    if matches.empty: return []
    out = [str(x).strip() for x in matches['rewritten_query'].tolist() if str(x).strip()]
    return out[:max_rewrites]

def get_qur_variants_filtered(original_query: str, qur_df: pd.DataFrame,
                              max_rewrites: int = 3, jaccard_min: float = 0.15) -> list:
    """Return [original] + filtered rewrites (deduped)."""
    variants = [original_query]
    if qur_df is None or qur_df.empty: return variants

    if 'strict_key' not in qur_df.columns and 'original_query' in qur_df.columns:
        qur_df['strict_key'] = qur_df['original_query'].apply(strict_clean)

    q_key = strict_clean(original_query)
    matches = qur_df[qur_df['strict_key'] == q_key] if 'strict_key' in qur_df.columns else pd.DataFrame()

    if matches.empty: return variants

    rewrites = []
    for r in matches['rewritten_query'].tolist():
        r = str(r).strip()
        if len(r) < 6: continue
        if jaccard_overlap(original_query, r) < jaccard_min: continue
        rewrites.append(r)

    variants.extend(rewrites[:max_rewrites])
    return list(dict.fromkeys(variants)) # stable dedupe

def rrf_fuse(lists_of_cids, k=60, weights=None):
    """Reciprocal Rank Fusion with optional list-weights."""
    scores = defaultdict(float)
    if weights is None: weights = [1.0] * len(lists_of_cids)
    for w, lst in zip(weights, lists_of_cids):
        for r, cid in enumerate(lst, 1):
            scores[cid] += float(w) * (1.0 / (k + r))
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# --- Core Logic Helpers (The "Brains") ---

def s5_hybrid_rrf_details(query: str, bm25_retriever, dense_retriever,
                          bm25_control_ids: list, candidate_set_size: int = 50, rrf_k: int = 60) -> dict:
    """Unified S5 logic: Computes RRF + Agreement + Margin Stats."""
    # 1. BM25
    q_tokens = tokenize(normalize_text(query))
    bm25_scores = bm25_retriever.get_scores(q_tokens)
    bm25_idxs = bm25_retriever.get_top_n(q_tokens, n=candidate_set_size)
    bm25_cids = [normalize_control_id(bm25_control_ids[i]).upper() for i in bm25_idxs]

    # 2. Dense
    try: dense_hits = dense_retriever.search(query, top_k=candidate_set_size)
    except: dense_hits = []
    dense_cids = [normalize_control_id(str(c)).upper() for c, _ in dense_hits]

    # 3. RRF
    rrf_scores = defaultdict(float)
    for r, cid in enumerate(bm25_cids, 1): rrf_scores[cid] += 1.0 / (rrf_k + r)
    for r, cid in enumerate(dense_cids, 1): rrf_scores[cid] += 1.0 / (rrf_k + r)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    fused_cids = [c for c, _ in fused]

    # 4. Stats (ensure floats)
    bm25_top1 = bm25_cids[0] if bm25_cids else ""
    dense_top1 = dense_cids[0] if dense_cids else ""

    bm25_s1 = float(bm25_scores[bm25_idxs[0]]) if bm25_idxs else 0.0
    bm25_s2 = float(bm25_scores[bm25_idxs[1]]) if len(bm25_idxs) > 1 else 0.0
    dense_s1 = float(dense_hits[0][1]) if dense_hits else 0.0
    dense_s2 = float(dense_hits[1][1]) if len(dense_hits) > 1 else 0.0

    fused_s1 = float(fused[0][1]) if len(fused) > 0 else 0.0
    fused_s2 = float(fused[1][1]) if len(fused) > 1 else 0.0
    margin_ratio = (fused_s1 - fused_s2) / max(fused_s1, 1e-9)
    agreement = int(bool(bm25_top1) and (bm25_top1 == dense_top1))

    return {
        "bm25_top1": bm25_top1, "bm25_s1": bm25_s1, "bm25_s2": bm25_s2,
        "dense_top1": dense_top1, "dense_s1": dense_s1, "dense_s2": dense_s2,
        "fused_cids": fused_cids, "rrf_scores": rrf_scores,
        "agreement": agreement, "margin_ratio": margin_ratio
    }

def s8_confidence_from_s5(s5: dict) -> float:
    agree = float(s5.get("agreement", 0))
    mr = float(s5.get("margin_ratio", 0.0))
    mr_norm = max(0.0, min(1.0, mr / max(S8_MARGIN_RATIO_STRONG, 1e-9)))
    return float(0.55 * agree + 0.45 * mr_norm)

def s7_heavy_retrieve_details(
    original_query,
    bm25_retriever,
    dense_retriever,
    bm25_control_ids,
    cross_enc,
    id_to_text_map,
    qur_df,
    candidate_set_size=50,
    rrf_k=60,
    max_rewrites=3,
    rewrite_weight=0.25,
    rewrite_jaccard_min=0.15,
    rerank_alpha=0.65,
    rerank_apply_min_margin_ratio=0.15,   # NEW: no-harm gate threshold
):
    """
    S7 logic: Weighted RRF + Safe Rerank Blending + No-Harm Rerank Gate.

    No-Harm Gate:
      - If reranker changes top1 but its margin is small, keep base ordering.
      - Prevents 'rank reversal' on ambiguous candidates.
    """

    variants = get_qur_variants_filtered(original_query, qur_df, max_rewrites, rewrite_jaccard_min)
    weights = [1.0] + [rewrite_weight] * (len(variants) - 1)

    # 1) Weighted RRF retrieval
    rrf_scores = defaultdict(float)
    best_text_cache = {}

    for q, w in zip(variants, weights):
        # BM25
        q_toks = tokenize(normalize_text(q))
        idxs = bm25_retriever.get_top_n(q_toks, n=candidate_set_size)
        for r, i in enumerate(idxs, 1):
            cid = normalize_control_id(bm25_control_ids[i]).upper()
            rrf_scores[cid] += float(w) * (1.0 / (rrf_k + r))
            if cid not in best_text_cache:
                best_text_cache[cid] = id_to_text_map.get(cid, "")

        # Dense
        try:
            hits = dense_retriever.search(q, top_k=candidate_set_size)
        except Exception:
            hits = []

        for r, (raw_cid, _) in enumerate(hits, 1):
            cid = normalize_control_id(str(raw_cid)).upper()
            rrf_scores[cid] += float(w) * (1.0 / (rrf_k + r))
            if hasattr(dense_retriever, "best_clause_text_for_control"):
                t = dense_retriever.best_clause_text_for_control(cid)
                if t:
                    best_text_cache[cid] = t

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:candidate_set_size]
    cids_rerank = [c for c, _ in fused]

    if not cids_rerank:
        return {"final_ranked_cids": [], "meta": {"note": "no_candidates", "rerank_applied": False}}

    # 2) Build cross-encoder pairs
    pairs = []
    valid_cids = []
    for cid in cids_rerank:
        txt = best_text_cache.get(cid) or id_to_text_map.get(cid, "")
        if txt:
            pairs.append([original_query, txt])
            valid_cids.append(cid)

    base_ranked_cids = valid_cids[:]  # base order = weighted RRF order

    # If we have no text, fallback to base order
    if not pairs:
        return {
            "final_ranked_cids": cids_rerank,
            "meta": {
                "note": "no_text_for_rerank",
                "rerank_applied": False,
                "num_variants": len(variants),
                "base_top1": cids_rerank[0] if cids_rerank else "",
            }
        }

    # 3) Cross-encoder scoring
    pred = cross_enc.predict(pairs, show_progress_bar=False)

    # Base scores (RRF) aligned to valid_cids
    base = np.array([rrf_scores.get(c, 0.0) for c in valid_cids], dtype=np.float32)
    base_n = normalize_scores_minmax(base)
    pred_n = normalize_scores_minmax(np.array(pred, dtype=np.float32))

    # Safe blending score
    final_scores = float(rerank_alpha) * base_n + (1.0 - float(rerank_alpha)) * pred_n

    # 4) Reranked order by blended score
    rerank_order = np.argsort(-final_scores)
    reranked_cids = [valid_cids[i] for i in rerank_order]

    # Compute rerank margin ratio (top1 vs top2)
    if len(rerank_order) >= 2:
        fs1 = float(final_scores[rerank_order[0]])
        fs2 = float(final_scores[rerank_order[1]])
        rerank_margin_ratio = (fs1 - fs2) / max(fs1, 1e-9)
    else:
        rerank_margin_ratio = 1.0

    # Compute base margin ratio (top1 vs top2) using base signal
    if len(base_n) >= 2:
        bs1 = float(base_n[0])
        bs2 = float(base_n[1])
        base_margin_ratio = (bs1 - bs2) / max(bs1, 1e-9)
    else:
        base_margin_ratio = 1.0

    base_top1 = base_ranked_cids[0] if base_ranked_cids else ""
    rerank_top1 = reranked_cids[0] if reranked_cids else ""

    # 5) No-Harm Rerank Gate:
    # If reranker flips top1 but has low margin, keep base order.
    rerank_applied = True
    final_ranked_cids = reranked_cids

    if (rerank_top1 != base_top1) and (rerank_margin_ratio < float(rerank_apply_min_margin_ratio)):
        rerank_applied = False
        final_ranked_cids = base_ranked_cids

    # Final margin depends on which ordering we used
    final_margin_ratio = rerank_margin_ratio if rerank_applied else base_margin_ratio
    final_top1 = final_ranked_cids[0] if final_ranked_cids else ""

    return {
        "final_ranked_cids": final_ranked_cids,
        "meta": {
            "num_variants": len(variants),
            "rerank_applied": rerank_applied,
            "rerank_alpha": float(rerank_alpha),
            "rerank_apply_min_margin_ratio": float(rerank_apply_min_margin_ratio),
            "base_top1": base_top1,
            "rerank_top1": rerank_top1,
            "final_top1": final_top1,
            "base_margin_ratio": float(base_margin_ratio),
            "rerank_margin_ratio": float(rerank_margin_ratio),
            "final_margin_ratio": float(final_margin_ratio),
        }
    }


# --- SYSTEM ENTRY POINTS ---

def system_s1_bm25(q, gold, bm25, doc_ids, top_n=10):
    q_toks = tokenize(normalize_text(q))
    idxs = bm25.get_top_n(q_toks, n=top_n)
    retrieved = [normalize_control_id(doc_ids[i]) for i in idxs]
    return {"rank": find_rank(gold, retrieved, top_n), "retrieved_ids": retrieved, "system_meta": {"sys": "S1"}}

def system_s2_dense(q, gold, dense, top_n=10):
    hits = dense.search(q, top_k=top_n)
    retrieved = [normalize_control_id(c) for c, s in hits]
    return {"rank": find_rank(gold, retrieved, top_n), "retrieved_ids": retrieved, "system_meta": {"sys": "S2"}}

def system_s3_rewrite_replace(q, gold, bm25, doc_ids, qur_df, top_n=10):
    rewrites = get_rewrites(q, qur_df, max_rewrites=1)
    q2 = rewrites[0] if rewrites else q
    return system_s1_bm25(q2, gold, bm25, doc_ids, top_n)

def system_s4_qur_rrf(q, gold, bm25, doc_ids, qur_df, top_n=10, rrf_k=60):
    variants = get_qur_variants_filtered(q, qur_df, max_rewrites=5, jaccard_min=S8_REWRITE_JACCARD_MIN)
    weights = [1.0] + [S8_REWRITE_WEIGHT] * (len(variants) - 1)

    lists = []
    for v in variants:
        q_toks = tokenize(normalize_text(v))
        idxs = bm25.get_top_n(q_toks, n=top_n)
        lists.append([normalize_control_id(doc_ids[i]) for i in idxs])

    fused = rrf_fuse(lists, k=rrf_k, weights=weights)
    retrieved = [c for c, _ in fused][:top_n]
    return {"rank": find_rank(gold, retrieved, top_n), "retrieved_ids": retrieved, "system_meta": {"sys": "S4", "n_vars": len(variants)}}

def system_s5_hybrid(q, gold, bm25, doc_ids, dense, top_n=10, candidate_k=50, rrf_k=60):
    """S5 baseline Hybrid RRF. S5 and S8 FAST path are mathematically identical."""
    details = s5_hybrid_rrf_details(q, bm25, dense, doc_ids, candidate_set_size=candidate_k, rrf_k=rrf_k)
    final = details["fused_cids"][:top_n]
    return {"rank": find_rank(gold, final, top_n), "retrieved_ids": final, "system_meta": {"sys": "S5"}}

def system_s6_rerank(q, gold, bm25, doc_ids, dense, id_to_text_map, top_n=10):
    res_s5 = system_s5_hybrid(q, gold, bm25, doc_ids, dense, top_n=50)
    cands = res_s5["retrieved_ids"]

    pairs = []
    valid_cands = []
    for cid in cands:
        txt = dense.best_clause_text_for_control(cid) if hasattr(dense, "best_clause_text_for_control") else None
        if not txt: txt = id_to_text_map.get(normalize_control_id(cid), "")
        if txt:
            pairs.append([q, txt])
            valid_cands.append(cid)

    if not pairs: return {"rank": 0, "retrieved_ids": [], "system_meta": {"sys": "S6", "note": "no_text"}}

    scores = cross_encoder.predict(pairs)
    reranked = [c for _, c in sorted(zip(scores, valid_cands), reverse=True)]
    final = reranked[:top_n]
    return {"rank": find_rank(gold, final, top_n), "retrieved_ids": final, "system_meta": {"sys": "S6"}}

def system_s7_compliancegpt(q, gold, bm25, doc_ids, dense, id_to_text_map, qur_df, top_n=10):
    heavy = s7_heavy_retrieve_details(
        q, bm25, dense, doc_ids, cross_encoder, id_to_text_map, qur_df,
        candidate_set_size=CANDIDATE_SET_SIZE, rrf_k=RRF_K_DEFAULT,
        rewrite_weight=S8_REWRITE_WEIGHT, rewrite_jaccard_min=S8_REWRITE_JACCARD_MIN, rerank_alpha=S8_RERANK_ALPHA
    )
    final = heavy["final_ranked_cids"][:top_n]
    meta = heavy.get("meta", {})
    meta["sys"] = "S7"
    return {"rank": find_rank(gold, final, top_n), "retrieved_ids": final, "system_meta": meta}

def system_s8_adaptive(
    q, gold,
    bm25, doc_ids,
    dense, id_to_text_map,
    qur_df,
    top_n=10,
    conf_threshold=None,
    heavy_min_margin_ratio=None
):
    """
    S8 Adaptive router:
    - FAST path uses S5 fused results
    - HEAVY path uses S7 heavy retrieve (rewrites + safe blend)

    IMPORTANT:
    - conf_threshold / heavy_min_margin_ratio are resolved at runtime
      (so grid search works correctly)
    """

    # Resolve thresholds at RUNTIME (not function-definition time)
    if conf_threshold is None:
        conf_threshold = float(S8_CONF_THRESHOLD)
    else:
        conf_threshold = float(conf_threshold)

    if heavy_min_margin_ratio is None:
        heavy_min_margin_ratio = float(S8_HEAVY_MIN_MARGIN_RATIO)
    else:
        heavy_min_margin_ratio = float(heavy_min_margin_ratio)

    # 1) FAST Path (S5 details)
    s5 = s5_hybrid_rrf_details(
        q, bm25, dense, doc_ids,
        candidate_set_size=CANDIDATE_SET_SIZE,
        rrf_k=RRF_K_DEFAULT
    )
    conf = float(s8_confidence_from_s5(s5))
    margin_ratio = float(s5.get("margin_ratio", 0.0))

    # FAST only when both signals are strong
    if (conf >= conf_threshold) and (margin_ratio >= heavy_min_margin_ratio):
        final = s5["fused_cids"][:top_n]
        return {
            "rank": find_rank(gold, final, top_n),
            "retrieved_ids": final,
            "system_meta": {
                "sys": "S8",
                "mode": "FAST",
                "conf": conf,
                "margin_ratio": margin_ratio,
                "conf_threshold": conf_threshold,
                "margin_threshold": heavy_min_margin_ratio,
            }
        }

    # 2) HEAVY Path (S7 heavy)
    heavy = s7_heavy_retrieve_details(
        q, bm25, dense, doc_ids,
        cross_encoder, id_to_text_map, qur_df,
        candidate_set_size=CANDIDATE_SET_SIZE,
        rrf_k=RRF_K_DEFAULT,
        rewrite_weight=S8_REWRITE_WEIGHT,
        rewrite_jaccard_min=S8_REWRITE_JACCARD_MIN,
        rerank_alpha=S8_RERANK_ALPHA
    )
    final = heavy["final_ranked_cids"][:top_n]
    if not final:
        final = s5["fused_cids"][:top_n]  # fallback

    meta = heavy.get("meta", {})
    meta.update({
        "sys": "S8",
        "mode": "HEAVY",
        "conf": conf,
        "margin_ratio": margin_ratio,
        "conf_threshold": conf_threshold,
        "margin_threshold": heavy_min_margin_ratio,
    })
    return {
        "rank": find_rank(gold, final, top_n),
        "retrieved_ids": final,
        "system_meta": meta
    }


In [8]:
# ==========================================================
# MASTER EVALUATOR (Unified Schema) — WITH nDCG & ODP
# ==========================================================

FINAL_COLUMNS = [
    "question_id",
    "question",
    "gold_control_id",
    "odp_required",
    "rank",
    "is_hit_at_1",
    "is_hit_at_5",
    "is_hit_at_10",
    "top_1_hit",
    "retrieved_control_ids",
    "system_meta",
]

def _stable_question_id(q: str, gold_cid: str) -> str:
    """Deterministic fallback hash if dataset lacks question_id."""
    import hashlib
    base = f"{str(q).strip()}||{normalize_control_id(str(gold_cid))}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()[:12]

def ndcg_at_k(ranks: List[int], k: int) -> float:
    """Mean nDCG@k over queries given their (1-based) gold ranks.

    - ranks: list of 1-based ranks for the gold item; 0 (or >k) means 'not found in top-k'.
    - With exactly 1 relevant item per query, per-query IDCG@k = 1.0, so nDCG@k reduces to:
        1 / log2(rank + 1) for hits within k, else 0.
    """
    if not ranks:
        return 0.0

    per_query = []
    for r in ranks:
        try:
            r_int = int(r)
        except Exception:
            r_int = 0

        if 1 <= r_int <= k:
            per_query.append(1.0 / math.log2(r_int + 1))
        else:
            per_query.append(0.0)

    return sum(per_query) / len(per_query)

def evaluate_system_unified(
    system_name: str,
    gold_df: pd.DataFrame,
    run_fn,
    out_csv: str,
    out_rpt: str = None,
    top_n: int = 10,
) -> pd.DataFrame:
    """Run a retrieval system over a dataset and save a unified-schema CSV."""

    # --- 1. Robust Column Normalization ---
    df = gold_df.copy()
    col_map = {}
    for c in df.columns:
        c_lower = c.lower().strip().replace(" ", "_").replace("-", "_")
        if c_lower in ["id", "qid", "question_id", "questionid", "q_id"]:
            col_map[c] = "question_id"
        elif c_lower in ["question", "query", "text", "description"]:
            col_map[c] = "question"
        elif c_lower in ["control_id", "gold_control_id", "control", "gold", "label"]:
            col_map[c] = "gold_control_id"
        elif c_lower in ["odp_required", "odp"]:
            col_map[c] = "odp_required"
    df = df.rename(columns=col_map)

    # --- 2. Evaluation Loop ---
    records = []
    n_total = len(df)
    print(f" [INFO] [{system_name}] Evaluating {n_total} queries...")

    rows = df.to_dict("records")

    for i, row in enumerate(rows, 1):
        q = str(row.get("question", "")).strip()
        gold = str(row.get("gold_control_id", "")).strip()
        qid = str(row.get("question_id", row.get("id", _stable_question_id(q, gold)))).strip()

        odp = row.get("odp_required", "N/A")
        if pd.isna(odp): odp = "N/A"

        # Run System
        try:
            res = run_fn(q, gold) or {}
        except Exception as e:
            print(f" [ERROR] QID {qid}: {e}")
            res = {}

        retrieved = res.get("retrieved_ids", res.get("retrieved_control_ids", [])) or []
        retrieved = [normalize_control_id(str(x)) for x in retrieved]

        r = int(res.get("rank", 0) or 0)
        meta = res.get("system_meta", {}) or {}

        rec = {
            "question_id": qid,
            "question": q,
            "gold_control_id": gold,
            "odp_required": str(odp),
            "rank": r,
            "is_hit_at_1": (r == 1),
            "is_hit_at_5": (1 <= r <= 5),
            "is_hit_at_10": (1 <= r <= 10),
            "top_1_hit": retrieved[0] if retrieved else "",
            "retrieved_control_ids": json.dumps(retrieved, ensure_ascii=False),
            "system_meta": json.dumps(meta, ensure_ascii=False),
        }
        records.append(rec)
        if (i % 25 == 0) or (i == n_total): print(f"..{i}", end="")

    print(" Done.")
    out_df = pd.DataFrame(records).reindex(columns=FINAL_COLUMNS)

    # --- 3. Save CSV ---
    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    out_df.to_csv(out_csv, index=False)
    print(f" Saved: {out_csv}")

    # --- 4. Generate Report (Overall + ODP Subset) ---
    if out_rpt:
        # A. Overall Stats
        ranks_all = out_df["rank"].tolist()
        n_all = len(ranks_all) or 1
        stats = {
            "Overall": {
                "n": n_all,
                "R@1": sum(1 for r in ranks_all if r==1)/n_all,
                "R@5": sum(1 for r in ranks_all if 1<=r<=5)/n_all,
                "R@10": sum(1 for r in ranks_all if 1<=r<=10)/n_all,
                "MRR@10": sum((1.0/r) for r in ranks_all if 1<=r<=10)/n_all,
                "nDCG@10": ndcg_at_k(ranks_all, 10)
            }
        }

        # B. ODP Subset Stats
        # Filter where odp_required is NOT "N/A" and NOT empty
        odp_df = out_df[~out_df["odp_required"].isin(["N/A", "nan", ""])]
        if not odp_df.empty:
            ranks_odp = odp_df["rank"].tolist()
            n_odp = len(ranks_odp)
            stats["ODP-Subset"] = {
                "n": n_odp,
                "R@1": sum(1 for r in ranks_odp if r==1)/n_odp,
                "R@5": sum(1 for r in ranks_odp if 1<=r<=5)/n_odp,
                "R@10": sum(1 for r in ranks_odp if 1<=r<=10)/n_odp,
                "MRR@10": sum((1.0/r) for r in ranks_odp if 1<=r<=10)/n_odp,
                "nDCG@10": ndcg_at_k(ranks_odp, 10)
            }

        # Write Report
        os.makedirs(os.path.dirname(out_rpt) or ".", exist_ok=True)

        # Overwrite report when starting a new system run (Rev5 is always first)
        rpt_mode = "a"
        if not os.path.exists(out_rpt):
            rpt_mode = "w"
        elif ("(Rev5)" in system_name) and ("ErrorBank" not in system_name):
            rpt_mode = "w"

        with open(out_rpt, rpt_mode, encoding="utf-8") as f:
            # Avoid leading blank line when overwriting
            if rpt_mode == "w":
                f.write(f"### {system_name}\n")
            else:
                f.write(f"\n### {system_name}\n")

            # Print table-like structure for each subset
            for subset, m in stats.items():
                f.write(f"**{subset} (n={m['n']})**\n")
                f.write(f"- Recall@1:  {m['R@1']:.4f}\n")
                f.write(f"- Recall@5:  {m['R@5']:.4f}\n")
                f.write(f"- Recall@10: {m['R@10']:.4f}\n")
                f.write(f"- MRR@10:    {m['MRR@10']:.4f}\n")
                f.write(f"- nDCG@10:   {m['nDCG@10']:.4f}\n")
                f.write("\n")
            f.write("---\n")

    return out_df

In [9]:
# ==========================================================
#  STEP 1: RUN BASELINE (S1) & ERROR BANK GENERATION
# ==========================================================

# 1. SETUP: Force Create Dir & CLEAN OLD REPORT
os.makedirs(S1_DIR, exist_ok=True)
s1_rpt_path = os.path.join(S1_DIR, "S1_bm25_report.md")
if os.path.exists(s1_rpt_path):
    os.remove(s1_rpt_path)
    print(f" [INFO] Deleted old report: {s1_rpt_path}")

# Helper to run Error Bank (Defined here so S1 can use it immediately)
def _run_error_bank_by_version(system_tag: str, run_rev5, run_rev4, out_dir: str):
    """Run system on error_bank_df split by version. Saves separate CSVs."""
    os.makedirs(out_dir, exist_ok=True)
    if error_bank_df.empty:
        print(f" [WARN] Error bank empty, skipping {system_tag} on error bank.")
        return

    # Filter by version
    eb5 = error_bank_df[error_bank_df.get('version', '') == 'rev5'].copy()
    eb4 = error_bank_df[error_bank_df.get('version', '') == 'rev4'].copy()

    # Run Rev5 Error Bank
    if len(eb5) > 0:
        evaluate_system_unified(
            f"{system_tag} (ErrorBank-Rev5)", eb5, run_rev5,
            os.path.join(out_dir, f"{system_tag}_error_bank_rev5_results.csv"),
            os.path.join(out_dir, f"{system_tag}_report.md"),
        )

    # Run Rev4 Error Bank
    if len(eb4) > 0:
        evaluate_system_unified(
            f"{system_tag} (ErrorBank-Rev4)", eb4, run_rev4,
            os.path.join(out_dir, f"{system_tag}_error_bank_rev4_results.csv"),
            os.path.join(out_dir, f"{system_tag}_report.md"),
        )

# 2. Run S1 Rev 5
s1_r5_df = evaluate_system_unified(
    "S1 BM25 (Rev5)", rev5_gold,
    lambda q,g: system_s1_bm25(q, g, bm25_rev5, rev5_ids),
    os.path.join(S1_DIR, "S1_bm25_rev5_results.csv"),
    s1_rpt_path
)

# 3. Run S1 Rev 4
s1_r4_df = evaluate_system_unified(
    "S1 BM25 (Rev4)", rev4_gold,
    lambda q,g: system_s1_bm25(q, g, bm25_rev4, rev4_ids),
    os.path.join(S1_DIR, "S1_bm25_rev4_results.csv"),
    s1_rpt_path
)

# 4. Conditional Error Bank Generation
if GENERATE_ERROR_BANK:
    print("\n [INFO] Generating Error Bank v2 from S1 failures...")

    # Filter Failures (Rank != 1)
    err_r5 = s1_r5_df[s1_r5_df["rank"] != 1].copy()
    err_r5["version"] = "rev5"
    err_r4 = s1_r4_df[s1_r4_df["rank"] != 1].copy()
    err_r4["version"] = "rev4"

    # Combine
    bank = pd.concat([err_r5, err_r4], ignore_index=True)
    if "question_id" in bank.columns:
        bank["id"] = bank["question_id"]
    else:
        bank["id"] = bank.index + 1

    bank["framework"] = "NIST_SP_800-53"
    bank["failure_category"] = ""
    bank["label_rationale"] = ""
    if "rank" in bank.columns:
        bank.rename(columns={"rank": "bm25_rank"}, inplace=True)

    # Save
    os.makedirs(os.path.dirname(ERROR_BANK_CSV), exist_ok=True)
    bank.to_csv(ERROR_BANK_CSV, index=False)
    print(f" [OK] Error Bank Generated: {len(bank)} queries. Saved to {ERROR_BANK_CSV}")

    # HOT RELOAD into memory
    error_bank_df = bank
    print(" [OK] Error Bank reloaded into memory.")

else:
    print(f"\n [INFO] Skipped Error Bank Generation (Flag=False). Using loaded file.")

# 5. Run S1 on Error Bank (For Table Completion)
TAG = "S1_bm25"
_run_error_bank_by_version(TAG,
                           lambda q,g: system_s1_bm25(q, g, bm25_rev5, rev5_ids),
                           lambda q,g: system_s1_bm25(q, g, bm25_rev4, rev4_ids),
                           S1_DIR)

print(" [OK] S1 Finished.")

 [INFO] Deleted old report: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_1_bm25/S1_bm25_report.md
 [INFO] [S1 BM25 (Rev5)] Evaluating 100 queries...
..25..50..75..100 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_1_bm25/S1_bm25_rev5_results.csv
 [INFO] [S1 BM25 (Rev4)] Evaluating 36 queries...
..25..36 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_1_bm25/S1_bm25_rev4_results.csv

 [INFO] Generating Error Bank v2 from S1 failures...
 [OK] Error Bank Generated: 37 queries. Saved to /content/drive/MyDrive/ComplianceGPT_v2/data/error_bank/error_bank_v1.csv
 [OK] Error Bank reloaded into memory.
 [INFO] [S1_bm25 (ErrorBank-Rev5)] Evaluating 24 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..24 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_1_bm25/S1_bm25_error_bank_rev5_results.csv
 [INFO] [S1_bm25 (ErrorBank-Rev4)] Evaluating 13 queries...
..13 Done.


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_1_bm25/S1_bm25_error_bank_rev4_results.csv
 [OK] S1 Finished.


# SECTION 2

In [10]:
# ==========================================================
#  STEP 2: RUN S2-S7 (CLEAN START)
# ==========================================================

# 1. CLEANUP OLD REPORTS
# We must delete the report files first, otherwise new runs just append to the bottom.
configs = [
    (S2_DIR, "S2_dense"),
    (S3_DIR, "S3_rewrite_only"),
    (S4_DIR, "S4_qur_rrf"),
    (S5_DIR, "S5_hybrid_rrf"),
    (S6_DIR, "S6_hybrid_rerank"),
    (S7_DIR, "S7_compliance_gpt"),
]

for d, tag in configs:
    os.makedirs(d, exist_ok=True)
    rpt = os.path.join(d, f"{tag}_report.md")
    if os.path.exists(rpt):
        os.remove(rpt)
        print(f" [INFO] Cleaned {tag} report.")

# (Helper _run_error_bank_by_version is reused from Step 1)

# --- S2: Dense ---
TAG = "S2_dense"
evaluate_system_unified(f"{TAG} (Rev5)", rev5_gold, lambda q,g: system_s2_dense(q,g,dense_retriever_rev5),
                        os.path.join(S2_DIR, f"{TAG}_rev5_results.csv"), os.path.join(S2_DIR, f"{TAG}_report.md"))
evaluate_system_unified(f"{TAG} (Rev4)", rev4_gold, lambda q,g: system_s2_dense(q,g,dense_retriever_rev4),
                        os.path.join(S2_DIR, f"{TAG}_rev4_results.csv"), os.path.join(S2_DIR, f"{TAG}_report.md"))
_run_error_bank_by_version(TAG, lambda q,g: system_s2_dense(q,g,dense_retriever_rev5),
                           lambda q,g: system_s2_dense(q,g,dense_retriever_rev4), S2_DIR)

# --- S3: Rewrite Only ---
TAG = "S3_rewrite_only"
evaluate_system_unified(f"{TAG} (Rev5)", rev5_gold, lambda q,g: system_s3_rewrite_replace(q,g,bm25_rev5,rev5_ids,qur_rev5_df),
                        os.path.join(S3_DIR, f"{TAG}_rev5_results.csv"), os.path.join(S3_DIR, f"{TAG}_report.md"))
evaluate_system_unified(f"{TAG} (Rev4)", rev4_gold, lambda q,g: system_s3_rewrite_replace(q,g,bm25_rev4,rev4_ids,qur_rev4_df),
                        os.path.join(S3_DIR, f"{TAG}_rev4_results.csv"), os.path.join(S3_DIR, f"{TAG}_report.md"))
_run_error_bank_by_version(TAG, lambda q,g: system_s3_rewrite_replace(q,g,bm25_rev5,rev5_ids,qur_errb_df),
                           lambda q,g: system_s3_rewrite_replace(q,g,bm25_rev4,rev4_ids,qur_errb_df), S3_DIR)

# --- S4: QUR RRF ---
TAG = "S4_qur_rrf"
evaluate_system_unified(f"{TAG} (Rev5)", rev5_gold, lambda q,g: system_s4_qur_rrf(q,g,bm25_rev5,rev5_ids,qur_rev5_df),
                        os.path.join(S4_DIR, f"{TAG}_rev5_results.csv"), os.path.join(S4_DIR, f"{TAG}_report.md"))
evaluate_system_unified(f"{TAG} (Rev4)", rev4_gold, lambda q,g: system_s4_qur_rrf(q,g,bm25_rev4,rev4_ids,qur_rev4_df),
                        os.path.join(S4_DIR, f"{TAG}_rev4_results.csv"), os.path.join(S4_DIR, f"{TAG}_report.md"))
_run_error_bank_by_version(TAG, lambda q,g: system_s4_qur_rrf(q,g,bm25_rev5,rev5_ids,qur_errb_df),
                           lambda q,g: system_s4_qur_rrf(q,g,bm25_rev4,rev4_ids,qur_errb_df), S4_DIR)

# --- S5: Hybrid RRF ---
TAG = "S5_hybrid_rrf"
evaluate_system_unified(f"{TAG} (Rev5)", rev5_gold, lambda q,g: system_s5_hybrid(q,g,bm25_rev5,rev5_ids,dense_retriever_rev5),
                        os.path.join(S5_DIR, f"{TAG}_rev5_results.csv"), os.path.join(S5_DIR, f"{TAG}_report.md"))
evaluate_system_unified(f"{TAG} (Rev4)", rev4_gold, lambda q,g: system_s5_hybrid(q,g,bm25_rev4,rev4_ids,dense_retriever_rev4),
                        os.path.join(S5_DIR, f"{TAG}_rev4_results.csv"), os.path.join(S5_DIR, f"{TAG}_report.md"))
_run_error_bank_by_version(TAG, lambda q,g: system_s5_hybrid(q,g,bm25_rev5,rev5_ids,dense_retriever_rev5),
                           lambda q,g: system_s5_hybrid(q,g,bm25_rev4,rev4_ids,dense_retriever_rev4), S5_DIR)

# --- S6: Hybrid Rerank ---
TAG = "S6_hybrid_rerank"
evaluate_system_unified(f"{TAG} (Rev5)", rev5_gold, lambda q,g: system_s6_rerank(q,g,bm25_rev5,rev5_ids,dense_retriever_rev5,rev5_id_to_text),
                        os.path.join(S6_DIR, f"{TAG}_rev5_results.csv"), os.path.join(S6_DIR, f"{TAG}_report.md"))
evaluate_system_unified(f"{TAG} (Rev4)", rev4_gold, lambda q,g: system_s6_rerank(q,g,bm25_rev4,rev4_ids,dense_retriever_rev4,rev4_id_to_text),
                        os.path.join(S6_DIR, f"{TAG}_rev4_results.csv"), os.path.join(S6_DIR, f"{TAG}_report.md"))
_run_error_bank_by_version(TAG, lambda q,g: system_s6_rerank(q,g,bm25_rev5,rev5_ids,dense_retriever_rev5,rev5_id_to_text),
                           lambda q,g: system_s6_rerank(q,g,bm25_rev4,rev4_ids,dense_retriever_rev4,rev4_id_to_text), S6_DIR)


 [INFO] Cleaned S2_dense report.
 [INFO] Cleaned S3_rewrite_only report.
 [INFO] Cleaned S4_qur_rrf report.
 [INFO] Cleaned S5_hybrid_rrf report.
 [INFO] Cleaned S6_hybrid_rerank report.
 [INFO] Cleaned S7_compliance_gpt report.
 [INFO] [S2_dense (Rev5)] Evaluating 100 queries...
..25..50..75..100 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_2_dense/S2_dense_rev5_results.csv
 [INFO] [S2_dense (Rev4)] Evaluating 36 queries...
..25..36 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_2_dense/S2_dense_rev4_results.csv
 [INFO] [S2_dense (ErrorBank-Rev5)] Evaluating 24 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..24 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_2_dense/S2_dense_error_bank_rev5_results.csv
 [INFO] [S2_dense (ErrorBank-Rev4)] Evaluating 13 queries...
..13 Done.


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_2_dense/S2_dense_error_bank_rev4_results.csv
 [INFO] [S3_rewrite_only (Rev5)] Evaluating 100 queries...
..25..50..75..100 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_3_rewrite_only/S3_rewrite_only_rev5_results.csv
 [INFO] [S3_rewrite_only (Rev4)] Evaluating 36 queries...
..25..36 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_3_rewrite_only/S3_rewrite_only_rev4_results.csv
 [INFO] [S3_rewrite_only (ErrorBank-Rev5)] Evaluating 24 queries...


...

..13 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_5_hybrid_rrf/S5_hybrid_rrf_error_bank_rev4_results.csv
 [INFO] [S6_hybrid_rerank (Rev5)] Evaluating 100 queries...
..25..50..75..100 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_6_hybrid_rerank/S6_hybrid_rerank_rev5_results.csv
 [INFO] [S6_hybrid_rerank (Rev4)] Evaluating 36 queries...
..25..36 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_6_hybrid_rerank/S6_hybrid_rerank_rev4_results.csv
 [INFO] [S6_hybrid_rerank (ErrorBank-Rev5)] Evaluating 24 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..24 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev5_results.csv
 [INFO] [S6_hybrid_rerank (ErrorBank-Rev4)] Evaluating 13 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..13 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev4_results.csv


In [11]:
# --- S7: ComplianceGPT ---
TAG = "S7_compliance_gpt"
evaluate_system_unified(f"{TAG} (Rev5)", rev5_gold, lambda q,g: system_s7_compliancegpt(q,g,bm25_rev5,rev5_ids,dense_retriever_rev5,rev5_id_to_text,qur_rev5_df),
                        os.path.join(S7_DIR, f"{TAG}_rev5_results.csv"), os.path.join(S7_DIR, f"{TAG}_report.md"))
evaluate_system_unified(f"{TAG} (Rev4)", rev4_gold, lambda q,g: system_s7_compliancegpt(q,g,bm25_rev4,rev4_ids,dense_retriever_rev4,rev4_id_to_text,qur_rev4_df),
                        os.path.join(S7_DIR, f"{TAG}_rev4_results.csv"), os.path.join(S7_DIR, f"{TAG}_report.md"))
_run_error_bank_by_version(TAG, lambda q,g: system_s7_compliancegpt(q,g,bm25_rev5,rev5_ids,dense_retriever_rev5,rev5_id_to_text,qur_errb_df),
                           lambda q,g: system_s7_compliancegpt(q,g,bm25_rev4,rev4_ids,dense_retriever_rev4,rev4_id_to_text,qur_errb_df), S7_DIR)

print(" [OK] S2-S7 finished.")

 [INFO] [S7_compliance_gpt (Rev5)] Evaluating 100 queries...
..25..50..75..100 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_rev5_results.csv
 [INFO] [S7_compliance_gpt (Rev4)] Evaluating 36 queries...
..25..36 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_rev4_results.csv
 [INFO] [S7_compliance_gpt (ErrorBank-Rev5)] Evaluating 24 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..24 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev5_results.csv
 [INFO] [S7_compliance_gpt (ErrorBank-Rev4)] Evaluating 13 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..13 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev4_results.csv
 [OK] S2-S7 finished.


System 8 uses a "Switching Mechanism." It tries to use the cheap Fast Path (S5) first. It only calls the Heavy Path (S7) if it detects low confidence.

In [12]:
# ==========================================================
#  STEP 3: RUN S8 (ADAPTIVE) - CLEAN START
# ==========================================================
os.makedirs(S8_DIR, exist_ok=True)
TAG = "S8_adaptive"

# Clean old report
rpt = os.path.join(S8_DIR, f"{TAG}_report.md")
if os.path.exists(rpt):
    os.remove(rpt)
    print(f" [INFO] Cleaned S8 report.")

# Rev 5
evaluate_system_unified(
    f"{TAG} (Rev5)", rev5_gold,
    lambda q, g: system_s8_adaptive(q, g, bm25_rev5, rev5_ids, dense_retriever_rev5, rev5_id_to_text, qur_rev5_df),
    os.path.join(S8_DIR, f"{TAG}_rev5_results.csv"), rpt
)

# Rev 4
evaluate_system_unified(
    f"{TAG} (Rev4)", rev4_gold,
    lambda q, g: system_s8_adaptive(q, g, bm25_rev4, rev4_ids, dense_retriever_rev4, rev4_id_to_text, qur_rev4_df),
    os.path.join(S8_DIR, f"{TAG}_rev4_results.csv"), rpt
)

# Error Bank
if not error_bank_df.empty:
    eb5 = error_bank_df[error_bank_df.get('version', '') == 'rev5'].copy()
    eb4 = error_bank_df[error_bank_df.get('version', '') == 'rev4'].copy()

    # Run Rev5 Error Bank
    if len(eb5) > 0:
        evaluate_system_unified(f"{TAG} (ErrorBank-Rev5)", eb5,
            lambda q, g: system_s8_adaptive(q, g, bm25_rev5, rev5_ids, dense_retriever_rev5, rev5_id_to_text, qur_errb_df),
            os.path.join(S8_DIR, f"{TAG}_error_bank_rev5_results.csv"), rpt
        )

    # Run Rev4 Error Bank
    if len(eb4) > 0:
        evaluate_system_unified(f"{TAG} (ErrorBank-Rev4)", eb4,
            lambda q, g: system_s8_adaptive(q, g, bm25_rev4, rev4_ids, dense_retriever_rev4, rev4_id_to_text, qur_errb_df),
            os.path.join(S8_DIR, f"{TAG}_error_bank_rev4_results.csv"), rpt
        )
else:
    print(" [INFO] Error Bank empty. Skipping S8 on Error Bank.")

print(" [OK] S8 finished.")

 [INFO] Cleaned S8 report.
 [INFO] [S8_adaptive (Rev5)] Evaluating 100 queries...
..25..50..75..100 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_8_adaptive/S8_adaptive_rev5_results.csv
 [INFO] [S8_adaptive (Rev4)] Evaluating 36 queries...
..25..36 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_8_adaptive/S8_adaptive_rev4_results.csv
 [INFO] [S8_adaptive (ErrorBank-Rev5)] Evaluating 24 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..24 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_8_adaptive/S8_adaptive_error_bank_rev5_results.csv
 [INFO] [S8_adaptive (ErrorBank-Rev4)] Evaluating 13 queries...


/tmp/ipython-input-2167568606.py:79: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rows = df.to_dict("records")


..13 Done.
 Saved: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/system_8_adaptive/S8_adaptive_error_bank_rev4_results.csv
 [OK] S8 finished.


S8 MICRO-EXPERIMENT

In [13]:
# ==========================================================
# SCIENTIFIC GRID SEARCH: SENSITIVITY ANALYSIS (S8)
# ==========================================================
import numpy as np
import pandas as pd

print(f"[INFO] Starting Sensitivity Analysis on Error Bank (n={len(error_bank_df)})...")

conf_range = np.linspace(0.40, 0.60, 5)
margin_range = np.linspace(0.005, 0.025, 5)

results = []
eb5 = error_bank_df[error_bank_df.get("version", "") == "rev5"].copy()
total = len(eb5)

target_hits = None  # optional
# target_hits = int(round(0.4815 * total))

print(f"[INFO] Grid Search Space: {len(conf_range) * len(margin_range)} combinations.")
print("-" * 65)
print(f"{'CONF':<10} | {'MARGIN':<10} | {'RECALL':<10} | {'HEAVY %':<10}")
print("-" * 65)

for conf in conf_range:
    for margin in margin_range:

        hits = 0
        heavy_calls = 0

        for _, row in eb5.iterrows():
            res = system_s8_adaptive(
                str(row["question"]),
                str(row["gold_control_id"]),
                bm25_rev5, rev5_ids,
                dense_retriever_rev5, rev5_id_to_text,
                qur_errb_df,
                conf_threshold=float(conf),
                heavy_min_margin_ratio=float(margin),
            )

            meta = res.get("system_meta", {})

            # runtime sanity check
            assert round(float(meta.get("conf_threshold", -1)), 6) == round(float(conf), 6)
            assert round(float(meta.get("margin_threshold", -1)), 6) == round(float(margin), 6)

            if res["rank"] == 1:
                hits += 1
            if meta.get("mode") == "HEAVY":
                heavy_calls += 1

        recall = hits / total if total else 0.0
        heavy_pct = (heavy_calls / total) * 100 if total else 0.0

        print(f"{conf:.2f}       | {margin:.3f}      | {recall:.4f}     | {heavy_pct:.1f}%")

        results.append({
            "conf": float(conf),
            "margin": float(margin),
            "recall": float(recall),
            "heavy_pct": float(heavy_pct),
            "hits": int(hits),
        })

df_res = pd.DataFrame(results)
best_hits = df_res["hits"].max()
candidates = df_res[df_res["hits"] == best_hits]
winner = candidates.sort_values(by="heavy_pct", ascending=True).iloc[0]

print("-" * 65)
print(f"OPTIMAL CONFIGURATION: Conf={winner['conf']:.2f}, Margin={winner['margin']:.3f}")
print(f"  Recall: {winner['recall']:.4f} (Hits: {int(winner['hits'])})")
print(f"  Heavy Usage: {winner['heavy_pct']:.1f}%")


[INFO] Starting Sensitivity Analysis on Error Bank (n=37)...
[INFO] Grid Search Space: 25 combinations.
-----------------------------------------------------------------
CONF       | MARGIN     | RECALL     | HEAVY %   
-----------------------------------------------------------------
0.40       | 0.005      | 0.5833     | 75.0%
0.40       | 0.010      | 0.5833     | 75.0%
0.40       | 0.015      | 0.5833     | 75.0%
0.40       | 0.020      | 0.5833     | 87.5%
0.40       | 0.025      | 0.5833     | 87.5%
0.45       | 0.005      | 0.5833     | 75.0%
0.45       | 0.010      | 0.5833     | 75.0%
0.45       | 0.015      | 0.5833     | 75.0%
0.45       | 0.020      | 0.5833     | 87.5%
0.45       | 0.025      | 0.5833     | 87.5%
0.50       | 0.005      | 0.5833     | 79.2%
0.50       | 0.010      | 0.5833     | 79.2%
0.50       | 0.015      | 0.5833     | 79.2%
0.50       | 0.020      | 0.5833     | 91.7%
0.50       | 0.025      | 0.5833     | 91.7%
0.55       | 0.005      | 0.5833     | 

Diagnose S8

In [14]:
eb5 = error_bank_df[error_bank_df.get("version","")=="rev5"].copy()

conf = 0.40
margin = 0.005

bad = []

for _, row in eb5.iterrows():
    q = str(row["question"])
    g = str(row["gold_control_id"])

    r7 = system_s7_compliancegpt(q, g, bm25_rev5, rev5_ids, dense_retriever_rev5, rev5_id_to_text, qur_errb_df)
    r8 = system_s8_adaptive(
        q, g,
        bm25_rev5, rev5_ids,
        dense_retriever_rev5, rev5_id_to_text,
        qur_errb_df,
        conf_threshold=conf,
        heavy_min_margin_ratio=margin
    )

    if (r7["rank"] == 1) and (r8["rank"] != 1):
        bad.append((q, g, r8.get("system_meta", {})))

print("S7 hit but S8 miss:", len(bad))
for q, g, meta in bad[:5]:
    print("\n---")
    print("Q:", q)
    print("Gold:", g)
    print("S8 meta:", meta)


S7 hit but S8 miss: 1

---
Q: What are the key elements that must be included in a contingency plan, and how should it be reviewed, distributed, and updated?
Gold: CP-2
S8 meta: {'sys': 'S8', 'mode': 'FAST', 'conf': 0.7249084249084251, 'margin_ratio': 0.04664224664224669, 'conf_threshold': 0.4, 'margin_threshold': 0.005}
